# Prompting vision-language models

<br><a target="_blank" href="https://colab.research.google.com/github/haukelicht/advanced_text_analysis/blob/main/notebooks/incontext_learning/llm_inference_basics_vision.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Learning goals

By the end of this notebook, you will be able to:

- Run a chat completion with vision-language model and image inputspipeline.
- Use a simple prompt to classify the sentiment of an image.

::: {.callout-note title="Inference"}

In the context of working with LLMs, _inference_ means generating predictions or responses from a trained language model based on the input you provide.
We use an existing model without updating its weights.

:::

## Setup

In [1]:
# note, we need torcvision (compatible with out required version of PyTorch)
%pip install torch==2.12.1 torchvision==0.27.1

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

# # If you want to only use model files already downloaded, uncomment:
# os.environ["HF_HUB_OFFLINE"] = "1"

import torch
import transformers
from transformers import pipeline
# disable verbosity
from transformers import logging
logging.set_verbosity_error()


print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

Transformers: 5.16.1
PyTorch: 2.12.1


We use `Qwen/Qwen2.5-0.5B-Instruct`, a smaller instruction-tuned model suitable for this local demonstration.
It is _very_ small but sufficient for demonstrating local inference with the Transformers pipeline.
<!-- It differs in size from the 72B model in the API notebook, so do not interpret differences in answers as effects of the client alone. -->

In [4]:
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

::: {.callout-warning title="Offline mode requires cached files"}

The model weights, tokenizer, and configuration files must already be cached.
If you need to download them, remove the offline-setting line and restart the kernel before running this notebook.
Alternatively, set `MODEL_ID` to a local folder containing the downloaded model.
No inference API token or API credit is needed for this public model.

:::

### Choose a compute device

A GPU can speed up generation.
CUDA supports compatible NVIDIA GPUs; MPS supports compatible Apple Silicon systems.
The CPU is the fallback and may be slower.

In [5]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

Device: mps


### Create the pipeline

Creating a pipeline loads the model into memory.
Unlike constructing an API client, this can take time and several gigabytes of memory.
We create it once and reuse it throughout the notebook.

In [6]:
vlm = pipeline(
    task="image-text-to-text",
    model=MODEL_ID,
    device=device,
    dtype="auto",
)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

::: {.callout-tip title="Device or memory problems?"}

If the accelerator does not support the checkpoint's precision, try `dtype=torch.float32`.
If accelerator memory is insufficient and you get a `RuntimeError` or out-of-memory error, try `device="cpu"`; the model still requires enough system memory.

:::

## First completion: the text-generation pipeline

### Define the conversation

As in the API notebook, a chat prompt is a list of messages with `role` and `content` keys:

- A `system` message gives the task instructions.
- A `user` message supplies the text to analyse.
- An `assistant` message represents a previous model response.

In [8]:
IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"

In [9]:
from IPython.display import display, HTML

display(HTML(f'<img src="{IMAGE_URL}" style="width: 40%; height: auto;">'))

In [10]:
task_instruction = """\
You classify the sentiment of an image. 

Identify the sentiment as positive, negative, or neutral. 

Only return the classification result without any explanation.\
"""

In [ ]:
messages = [
    {"role": "system", "content": task_instruction},
    {
        "role": "user",
        "content": [
            {"type": "image", "image": IMAGE_URL},
            {"type": "text", "text": "Classify the sentiment of this image."},
        ],
    }
]

::: {.callout-tip title="Using system and user messages"}

Separating instructions from the text makes the prompt easier to inspect and reuse.
Passing this list to the pipeline activates chat formatting automatically.
We do not need to manually insert role delimiters or special tokens.

:::

### Generating a response

Our first call uses the pipeline as-is.

In [11]:
result = vlm(messages)

### Inspect the response step by step

The pipeline returns a list of dictionaries. 
For this single conversation and one returned completion, the list has one element.

In [12]:
print("Type:", type(result))
print("Number of completions:", len(result))
print("First completion type:", type(result[0]))
print("Keys:", list(result[0].keys()))

Type: <class 'list'>
Number of completions: 1
First completion type: <class 'dict'>
Keys: ['input_text', 'generated_text']


By default, `generated_text` contains the conversation including the newly generated assistant message.

In [13]:
print(*result[0]["generated_text"], sep="\n")

{'role': 'system', 'content': 'You classify the sentiment of an image. \n\nIdentify the sentiment as positive, negative, or neutral. \n\nOnly return the classification result without any explanation.'}
{'role': 'user', 'content': [{'type': 'image', 'image': 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg'}, {'type': 'text', 'text': 'Classify the sentiment of this image.'}]}
{'role': 'assistant', 'content': 'positive'}


The last message is the **assistant's answer** with the classification.

The index `-1` selects the last element of a Python list.

In [14]:
assistant_message = result[0]["generated_text"][-1]
print(assistant_message)
print(assistant_message["content"])

{'role': 'assistant', 'content': 'positive'}
positive


::: {.callout-tip title="Congratulations"}

You have used a locally running LLM to complete a text annotation task!
The returned text is the model's classification choice, expressed as generated text.

:::

## use through API

This section demonstrates how to use the vision-language model through an API, rather than running it locally.

In [3]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

model_id = "Qwen/Qwen2.5-VL-72B-Instruct:ovhcloud"

In [ ]:
messages = [
    {"role": "system", "content": task_instruction},
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": IMAGE_URL}
                # alternatively: base64 encoded image data can be used here instead of a URL
                # "image_data": {"data": image_uri}
                
            },
            {"type": "text", "text": "Classify the sentiment of this image."},
        ],
    }
]

In [19]:
completion = client.chat.completions.create(
    model=model_id,
    messages=messages
)

print(completion.choices[0].message.content)

positive


In [22]:
# base64 encoded image from URL
from base64 import b64encode
import requests

response = requests.get(IMAGE_URL)
encoded_image = b64encode(response.content).decode("utf-8")
image_uri = f"data:image/jpeg;base64,{encoded_image}"

In [25]:
messages = [
    {"role": "system", "content": task_instruction},
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": image_uri}
                
            },
            {"type": "text", "text": "Classify the sentiment of this image."},
        ],
    }
]

completion = client.chat.completions.create(
    model=model_id,
    messages=messages
)

print(completion.choices[0].message.content)

positive
